# Budgerigar：Colab 数据准备与真实审计

本 notebook 是阶段 1 的真实验证入口。CMU ARCTIC 从官方 Festvox 归档下载并校验哈希；ESD 仅供研究使用，请先阅读 [ESD 官方下载页](https://hltsingapore.github.io/ESD/download.html) 并自行把官方数据放入 Colab 或 Drive。

In [ ]:
#@title 1. 获取项目（填写 Git 仓库地址，或先把项目上传到 /content/Budgerigar）
REPO_URL = "" #@param {type:"string"}
REPO_DIR = "/content/Budgerigar"
from pathlib import Path
import subprocess, sys
if not Path(REPO_DIR).is_dir():
    if not REPO_URL:
        raise ValueError("请填写 REPO_URL，或先将项目上传到 /content/Budgerigar")
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
print("Project ready:", REPO_DIR)

In [ ]:
#@title 2. 环境与许可确认
ACCEPT_DATASET_TERMS = False #@param {type:"boolean"}
USE_DRIVE = True #@param {type:"boolean"}
if not ACCEPT_DATASET_TERMS:
    raise RuntimeError("请阅读数据集官方条款，并勾选 ACCEPT_DATASET_TERMS")
import os, platform
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_ROOT = Path('/content/drive/MyDrive/Budgerigar')
else:
    WORK_ROOT = Path('/content/budgerigar_work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run(["nvidia-smi"], check=False)
print(platform.platform(), WORK_ROOT)

In [ ]:
#@title 3A. 下载并索引 CMU ARCTIC（推荐先用两位说话人 smoke test）
CMU_SPEAKERS = "bdl slt" #@param {type:"string"}
from budgerigar.data_prep import download_cmu_arctic, build_cmu_manifest, write_manifest
cmu_root = WORK_ROOT / 'data' / 'cmu_arctic'
download_cmu_arctic(cmu_root, CMU_SPEAKERS.split())
cmu_manifest = write_manifest(build_cmu_manifest(cmu_root), WORK_ROOT / 'manifests' / 'cmu_arctic.jsonl')
print(cmu_manifest)

In [ ]:
#@title 3B. 可选：索引已手动取得的官方 ESD
BUILD_ESD = False #@param {type:"boolean"}
ESD_ROOT = "/content/drive/MyDrive/Budgerigar/data/ESD" #@param {type:"string"}
if BUILD_ESD:
    from budgerigar.data_prep import build_esd_manifest
    esd_manifest = write_manifest(build_esd_manifest(ESD_ROOT), WORK_ROOT / 'manifests' / 'esd.jsonl')
    print(esd_manifest)

In [ ]:
#@title 4. 真实数据审计并保存报告
import json
from dataclasses import asdict
from budgerigar.manifest import load_manifest, audit_manifest
reports = {}
for manifest in [cmu_manifest] + ([esd_manifest] if BUILD_ESD else []):
    report = audit_manifest(load_manifest(manifest), check_audio=True)
    reports[Path(manifest).stem] = asdict(report)
    print(Path(manifest).name, report)
    if not report.ok:
        raise RuntimeError(f'数据审计失败：{manifest}')
report_path = WORK_ROOT / 'reports' / 'data_audit.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(reports, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', report_path)

In [ ]:
#@title 5. 保存 Colab 运行元数据
from budgerigar.experiment import write_run_metadata
metadata_path = write_run_metadata(WORK_ROOT / 'reports' / 'run_metadata.json', cmu_manifest)
print(metadata_path.read_text(encoding='utf-8'))